# 34 · 生成评估：Faithfulness / 框架 / 基准

> 检索再好，回答满嘴跑火车照样完蛋。生成质量用**忠实度与相关性**衡量——最主流的方式是**让 LLM 当裁判**（LLM-as-a-Judge）。

**本文件覆盖知识点**：Faithfulness / Answer Relevance / Context Relevance / LLM-as-a-Judge / RAGAS / DeepEval / TruLens / LangSmith / 数据集 MS MARCO / BEIR

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 三个最核心的生成指标

| 指标 | 英文 | 问的是 |
|------|------|--------|
| **忠实度** | Faithfulness | 回答是否**忠于检索到的上下文**、没瞎编 |
| **答案相关性** | Answer Relevance | 回答是否**答在点上**（不是废话也相关） |
| **上下文相关性** | Context Relevance | 检索到的上下文**是否用上了/有没有噪声** |

> Faithfulness 是防幻觉的关键闸门：回答里的每个论断都应能溯源到某个检索片段。

In [ ]:
# 用一个极简"断言列表"思想演示 Faithfulness 判定（教学版）
# 真实实现 = LLM 把回答拆成若干断言，再逐一判断每个断言能否被上下文支持

context = ['星云支持公有云与私有化两种部署方式']
answer  = '星云既支持公有云也支持私有化部署，而且性能是友商两倍。'

claims = ['星云支持公有云部署',            # 支持 → 有据
          '星云支持私有化部署',            # 支持 → 有据
          '性能是友商两倍']               # 上下文无此信息 → 幻觉

supported = sum(1 for c in claims if any(c in ctx for ctx in context))
faithfulness = supported / len(claims)
print(f'断言被支持数: {supported}/{len(claims)}')
print(f'Faithfulness = {faithfulness:.2f}  ← “性能两倍”无据导致扣分')
print('\n生产做法: 让 LLM 拆断言 → 逐条与上下文比对 → 输出 0-1 分数')

In [ ]:
# 知识点·真调说明：Faithfulness 判定 —— 让模型把回答拆成可核验断言，再逐条对照上下文判“有据”
import json as _json
out = _llm_live(
    prompt='把下面这段“模型回答”拆成若干条可独立核验的事实断言，再对照“检索到的上下文”，'
           '逐条标记 supported=true/false（上下文没有依据就标 false）。\n'
           '上下文: 星云支持公有云与私有化两种部署方式。\n'
           '模型回答: 星云既支持公有云也支持私有化部署，而且性能是友商两倍。\n'
           '只输出 JSON：{"claims": [{"claim": "<断言>", "supported": true 或 false}]}',
    system='你是 RAG 忠实度(Faithfulness)评估器。判定只依据给定上下文，上下文没提到的内容一律标 false，'
           '不自行补充知识。只输出 JSON，禁止输出其它文字。',
    fallback='未配置 Key 的固定样例：\n'
             '{"claims": [{"claim": "星云支持公有云部署", "supported": true}, '
             '{"claim": "星云支持私有化部署", "supported": true}, '
             '{"claim": "星云性能是友商两倍", "supported": false}]}',
    temperature=0.1,
)
if out is None:
    out = ('{"claims": [{"claim": "星云支持公有云部署", "supported": true}, '
           '{"claim": "星云支持私有化部署", "supported": true}, '
           '{"claim": "星云性能是友商两倍", "supported": false}]}')
    print('（以上为固定样例；下面用样例演示程序化汇总）')
try:
    data = _json.loads(out)
    claims = data['claims']
    supported = sum(1 for c in claims if c['supported'])
    print('模型把回答拆成 %d 条断言，其中 %d 条在上下文里有依据。' % (len(claims), supported))
    print('Faithfulness = %.2f' % (supported / len(claims)))
    unsup = [c['claim'] for c in claims if not c['supported']]
    if unsup:
        print('被判“无依据”的断言（幻觉）：', '；'.join(unsup))
except Exception as e:
    print('未解析成 JSON：', e, '—— 说明需在 prompt 里收紧输出格式。')
print('→ 先拆断言、再逐条核验，幻觉到底出在哪个论断一目了然——这是 RAGAS/DeepEval 计算 Faithfulness 的真实机制。')

In [ ]:
# LLM-as-a-Judge：让裁判模型按评分卡打分（骨架，配置 .env 后可用）
from dotenv import load_dotenv; load_dotenv()
import os
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def judge(question, context, answer):
    """让 qwen 按 3 个维度打分，输出 JSON"""
    from dashscope import Generation
    p = f"""你是 RAG 评估裁判。
问题: {question}\n上下文: {context}\n回答: {answer}
分别对 faithfulness(忠于上下文程度)、answer_relevance(切题程度)、context_relevance(上下文相关程度) 打分 0-1，只输出 JSON 如 {{"faithfulness":0.9,...}}"""
    r = Generation.call(model='qwen-plus', messages=[{'role':'user','content':p}],
                        api_key=API_KEY, result_format='message')
    return r.output.choices[0].message.content

if API_KEY and '你的' not in API_KEY:
    print('裁判打分:', judge('星云支持私有化吗', ['星云支持公有云与私有化两种部署'], '支持，也支持公有云。'))
else:
    print('LLM-as-a-Judge 骨架就绪。配置 DASHSCOPE_API_KEY 后，同一套评分卡可批量评估你的 RAG。')

## 2. 评估框架

| 框架 | 特色 | 定位 |
|------|------|------|
| **RAGAS** | 指标体系最贴 RAG（faithfulness/context precision…） | 离线指标库 |
| **DeepEval** | pytest 风格、断言式 | 单测化评估 |
| **TruLens** | 反馈函数 + 可视化追踪 | 反馈/追踪 |
| **LangSmith** | 线上 trace + 标注 + 数据集回归 | 生产观测 |
| **RAGChecker** | 细粒度诊断（噪声敏感度等） | 深度诊断 |

> 同一套**评测集**固定后，框架只是帮你把“指标算出来”。

## 3. 开源数据集与基准

| 数据集 | 内容 | 用途 |
|--------|------|------|
| **MS MARCO** | 必应搜索真实查询+段落 | 检索/排序基准 |
| **BEIR** | 18 个异构任务合集 | 零样本泛化能力 |
| **Natural Questions** | Google 搜索问答 | 开放域问答 |
| **KILT / TriviaQA** | 知识密集任务 | 知识型 RAG |

## 小结

- 生成三指标：**Faithfulness / Answer Relevance / Context Relevance**；
- 主流做法是 **LLM-as-a-Judge**，打分卡要固定、可复现；
- 框架（RAGAS 等）与基准（MS MARCO/BEIR）解决“怎么算”“在哪比”。